# 轨迹推断与伪时间排序


## 安装所需库


In [ ]:
# Download a script from GitHub that configures package installation 
# for R from the system's package manager (apt).
download.file("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_jammy.sh",
              "add_cranapt_jammy.sh")

# Grant execution permissions to the downloaded script
Sys.chmod("add_cranapt_jammy.sh", "0755")

# Execute the script to set up R package installation via apt
system("./add_cranapt_jammy.sh")

# Enable bspm (Bridge to System Package Manager), which allows installing R packages from the system’s package manager
bspm::enable()

# Disable version checking for bspm to prevent compatibility issues
options(bspm.version.check=FALSE)

# Remove the script after execution to keep the environment clean
system("rm add_cranapt_jammy.sh")


我们将创建一个 R 函数，用于执行系统调用。


In [2]:
# Define a function to execute shell commands and capture their output
shell_call <- function(command, ...) {
  # Execute the command in the system shell and capture the output
  result <- system(command, intern = TRUE, ...)
  
  # Print the output in a readable format
  cat(paste0(result, collapse = "\n"))
}

安装所需库。


In [ ]:
# Install the R.utils package, which provides additional utility functions for the next steps
install.packages("R.utils")

# Install specific versions of Seurat Wrappers and Seurat Data from GitHub
remotes::install_github('satijalab/seurat-wrappers@d28512f804d5fe05e6d68900ca9221020d52cf1d', upgrade=F)
remotes::install_github('satijalab/seurat-data')

# Check if BiocManager is installed; if not, install it for managing Bioconductor packages
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager", quiet = T)

# Install additional required packages
install.packages("harmony")  # Harmony for batch correction
BiocManager::install("clusterProfiler", update = T, ask=F, force=T) # For Functional Enrichment Analysis
BiocManager::install("destiny", update = F) # For Diffusion maps for single-cell data
remotes::install_github('cole-trapnell-lab/monocle3') # Install Monocle3 for trajectory inference

# Optional: Install a specific version of the Matrix package (commented out)
# install.packages("https://cran.r-project.org/src/contrib/Archive/Matrix/Matrix_1.5-3.tar.gz", repos=NULL, type="source")


## 引言


在发育和生命过程中，细胞会不断在不同功能状态之间转换。在这些转换过程中，基因表达会动态变化：有些基因被激活，另一些基因被沉默。单细胞 RNA 测序（scRNA-seq）允许研究者以高分辨率捕捉这些动态变化。Monocle3 等计算工具可以利用 scRNA-seq 数据重建细胞轨迹，帮助我们理解细胞如何随时间推进到不同状态。这种方法特别适用于研究细胞分化、疾病进展和细胞重编程。

在本教程中，我们将学习如何使用 Monocle3 推断细胞轨迹，并估计 pseudotime（伪时间）。伪时间是衡量细胞沿发育路径相对进程的指标。通过分析单细胞数据，我们可以描绘细胞如何在不同功能状态之间演变，并识别驱动这些转换的关键基因。

本教程受到以下既有指南和研究的启发，并在其基础上展开。这些资料展示了轨迹推断在单细胞生物学中的作用。

- [Monocle3 原始教程](https://cole-trapnell-lab.github.io/monocle3/docs/trajectories/)
- [Stuart Lab 的 Seurat 与 Monocle3 联合教程](https://stuartlab.org/signac/articles/monocle.html)
- [Mahima Bose 的 Seurat 与 Monocle3 联合教程](https://rpubs.com/mahima_bose/Seurat_and_Monocle3_p)

![Monocle3](https://cole-trapnell-lab.github.io/monocle3/images/manual_images/embryo_pr_graph_by_pseudotime.png)


In [ ]:
# Load the necessary libraries for single-cell RNA-seq analysis
library(monocle3)      # Trajectory inference
library(Seurat)        # Single-cell analysis framework
library(SeuratData)    # Preprocessed single-cell datasets
library(SeuratWrappers) # Additional Seurat functionalities
library(patchwork)     # Plot composition
library(harmony)       # Batch effect correction
library(ggplot2)       # Data visualization

## 加载数据


这里加载我们感兴趣的数据集，用于整合多个 scRNA-seq 样本。

本教程演示如何对齐 [Kang et al., 2017](https://www.nature.com/articles/nbt.4042) 研究中的两组外周血单个核细胞（PBMC）。在该实验中，PBMC 被分成对照组和刺激组；刺激组接受了 interferon-beta 处理。这种刺激导致细胞类型特异性的基因表达变化。因此，在分析数据时，细胞不仅会按照其生物学身份（细胞类型）聚类，也会受到刺激条件影响而聚类。这给联合分析带来挑战，因为表达模式差异可能掩盖两组中相同细胞类型之间的内在相似性。

通过整合这些数据集，我们希望校正批次效应和条件特异性变化，从而更准确地比较两组之间共有的生物学特征。


In [ ]:
# Download and install the "ifnb" dataset, which contains single-cell RNA-seq data
InstallData("ifnb")

In [ ]:
# Load the previously installed "ifnb" dataset
LoadData("ifnb")

In [7]:
# Store the loaded dataset in a new variable called 'testdata' 
# This allows us to modify the dataset while keeping the original one intact
testdata <- ifnb

In [ ]:
# Ensure the Seurat object is updated to the latest format
testdata <- UpdateSeuratObject(object = testdata)

# Display an overview of the dataset using dplyr::glimpse()
testdata %>% dplyr::glimpse()

## 数据处理


在运行 Monocle3 之前，我们先执行常规数据处理、整合、批次校正和聚类。


In [ ]:
# Here is the step by step processing
testdata <- Seurat::NormalizeData(testdata, verbose = FALSE) %>%  # Normalize gene expression data
            FindVariableFeatures(selection.method = "vst", nfeatures = 2000) %>%  # Identify 2000 most variable genes
            ScaleData(verbose = FALSE) %>%  # Standardize and center the data
            RunPCA(npcs = 30, verbose = FALSE) %>%  # Perform Principal Component Analysis (PCA) with 30 components
            RunHarmony("stim", plot_convergence = FALSE) %>%  # Batch correction using Harmony
            RunUMAP(reduction = "harmony", dims = 1:30) %>%  # Perform UMAP clustering using Harmony-corrected data
            FindNeighbors(reduction = "harmony", dims = 1:30) %>%  # Compute nearest neighbors for clustering
            FindClusters(resolution = 0.5)  # Cluster cells using Louvain algorithm

# The argument 'verbose = FALSE' suppresses output messages to keep the console clean

UMAP 图如下所示：


In [10]:
# Create a UMAP plot with cluster labels based on 'seurat_annotations'
scPlot <- DimPlot(testdata, label = TRUE, group.by = 'seurat_annotations')

# Display the plot
scPlot

# Optional: Save the plot as an image (commented out)
# ggsave("01-DimPlot.png", plot = scPlot, bg = "white")

## 运行 Monocle3


要使用 Monocle3 分析细胞轨迹，首先需要把 Seurat 对象转换为 Monocle3 能够处理的格式。这里使用 SeuratWrappers 包中的 `as.cell_data_set()` 函数完成转换。该函数会把 Seurat 对象转换为 `CellDataSet` 对象，作为 Monocle3 轨迹推断算法的输入。完成转换后，该对象即可用于构建发育轨迹、推断伪时间，并分析细胞状态转换。


In [ ]:
# Convert Seurat object into a Monocle3-compatible cell_data_set (cds)
cds <- as.cell_data_set(testdata)

# Store gene names as metadata in the cell dataset
fData(cds)$gene_short_name <- rownames(fData(cds))

In [ ]:
# Get an overview of the structure of the cell dataset (cds)
cds %>% dplyr::glimpse()

### **推断轨迹**

Monocle3 会通过聚类算法判断细胞应被放在同一条轨迹中，还是分配到不同轨迹中。在这个过程中，每个细胞不仅会被分入某个 cluster，也会被分配到一个 partition；partition 表示数据集中彼此相对独立的区域。构建轨迹时，Monocle3 会把每个 partition 视为一条独立轨迹。在这一步中，我们先用 `cluster_cells()` 函数对细胞进行聚类，识别具有生物学意义的分组；然后使用 `learn_graph()` 推断轨迹结构，从而可视化并分析细胞所遵循的发育路径。


In [ ]:
cds <- cluster_cells(cds = cds,  # Perform clustering on the cell dataset
                     reduction_method = "UMAP",  # Use UMAP for dimensionality reduction
                     cluster_method = 'louvain') %>%  # Apply Louvain algorithm for clustering
       learn_graph(use_partition = T)  # Learn the cellular trajectory graph


接下来，可视化推断出的轨迹，观察细胞如何沿发育路径组织。

在图中，黑色线条表示推断轨迹的结构，形成连接相关细胞的图。如果该图没有完全连通，说明不同 partition 中的细胞可能遵循不同发育路径。

图中的特殊点用带编号的圆圈标记。
分支末端的浅灰色圆圈对应不同细胞命运，也就是轨迹中可能的最终状态。
黑色圆圈表示分支点，细胞可以从这里走向不同发育方向。

可以通过 `plot_cells()` 中的 `label_leaves` 和 `label_branch_points` 参数自定义可视化。这些选项控制是否在图中标记细胞命运和分支点。需要注意，圆圈中的数字只是参考标记，并不代表特定生物学含义。


In [ ]:
# Generate a UMAP plot showing cell clusters along with trajectory landmarks  
scPlot <- plot_cells(cds, 
                     color_cells_by = "cluster",  # Color cells by cluster assignment  
                     label_groups_by_cluster = FALSE,  # Do not label clusters  
                     label_branch_points = TRUE,  # Label branch points in the trajectory  
                     label_roots = TRUE,  # Label the root cells in the trajectory  
                     label_leaves = TRUE,  # Label the leaf nodes in the trajectory  
                     group_label_size = 5)  # Set the font size for labels  

# Display the plot  
scPlot  

# Optional: Save the plot as an image  
# ggsave("02-plot_cells.png", plot = scPlot, bg = "white", width = 9, height = 9, dpi = 600)

移除标签后，可以得到更简洁的可视化结果。


In [ ]:
# Generate a UMAP plot showing cell clusters without trajectory landmarks  
scPlot <- plot_cells(cds, 
                     color_cells_by = "cluster",  # Color cells by cluster assignment  
                     label_groups_by_cluster = FALSE,  # Do not label clusters  
                     label_branch_points = FALSE,  # Do not label branch points  
                     label_roots = FALSE,  # Do not label root cells  
                     label_leaves = FALSE,  # Do not label leaf nodes  
                     group_label_size = 5)  # Set the font size for labels  

# Display the plot  
scPlot  

# Optional: Save the plot as an image  
# ggsave("03-plot_cells.png", plot = scPlot, bg = "white", width = 9, height = 9, dpi = 600)

**推断伪时间**

伪时间根据基因表达相似性估计细胞在某个生物学过程中的进程。Monocle3 会沿轨迹对细胞排序，并为每个细胞分配一个伪时间值，表示它相对于定义起点的位置。靠近 root 的细胞伪时间较低，沿路径更远的细胞伪时间较高，表示状态更晚或更推进。这有助于建模分化过程，并识别随时间变化的关键调控事件。

Monocle3 使用伪时间沿学习得到的轨迹对细胞排序。伪时间是一种抽象的进程度量，由细胞到轨迹起点之间沿最短路径的距离决定。轨迹长度对应细胞从初始状态到最终状态经历的总转录变化。


比较带注释的 UMAP 和 Monocle3 轨迹后，我们可以定义哪些 Monocle3 cluster 作为推断分化方向的 root。


In [ ]:
# Set plot size (optional)  
# options(repr.plot.height = 9, repr.plot.width = 16)

# Create a UMAP plot with cluster annotations from Seurat  
gumap <- DimPlot(testdata, label = TRUE, group.by = 'seurat_annotations')

# Create a Monocle3 plot showing clusters without trajectory landmarks  
gcluster <- plot_cells(cds, 
                       color_cells_by = "cluster",  
                       label_groups_by_cluster = FALSE,  
                       label_branch_points = FALSE,  
                       label_roots = FALSE,  
                       label_leaves = FALSE,  
                       group_label_size = 5)

# Combine both plots into a single figure  
scPlot <- gumap + gcluster + theme(aspect.ratio = 1)

# Display the combined plot  
scPlot  

# Optional: Save the plot as an image  
# ggsave("04-DimPlot-plot_cells.png", plot = scPlot, bg = "white", width = 18, height = 9, dpi = 600)

然后，使用下面的命令选择轨迹中的 root cells 或起始状态，并推断其他每个细胞的伪时间。


In [ ]:
# Assign a pseudotemporal order to cells using specific clusters as root cells  
cds <- order_cells(cds, 
                   reduction_method = "UMAP",  # Use UMAP for trajectory inference  
                   root_cells = colnames(cds[, clusters(cds) %in% c(3, 15, 9, 22)]))  # Specify root clusters  

随后，可以根据伪时间为轨迹上的细胞着色。


In [ ]:
# Set plot size (optional)  
options(repr.plot.height = 7, repr.plot.width = 7)

# Generate a UMAP plot with cells colored by pseudotime  
scPlot1 <- plot_cells(cds, 
                      color_cells_by = "pseudotime",  # Color cells based on their pseudotime  
                      label_groups_by_cluster = FALSE,  
                      label_branch_points = FALSE,  
                      label_roots = FALSE,  
                      label_leaves = FALSE,  
                      group_label_size = 5)

# Display the plot  
scPlot1  

# Save the plot as an image  
ggsave("05-plot_cells.png", plot = scPlot1, bg = "white", width = 9, height = 9, dpi = 600)

Seurat cluster、轨迹和伪时间的联合 UMAP 表示。


In [ ]:
# Set plot size (optional)  
# options(repr.plot.height=6, repr.plot.width=16)

# Combine previous plots (Seurat UMAP, Monocle3 clusters, and pseudotime)  
scPlot <- gumap + gcluster + scPlot1

# Display the combined figure  
scPlot  

# Save the combined plot as an image  
ggsave("06-Multiple_plots.png", plot = scPlot, bg = "white", width = 27, height = 9, dpi = 600)

我们可以按照 cluster 对应的伪时间，对 Seurat cluster 进行排序。


In [ ]:
# Set plot size (optional)  
# options(repr.plot.height=7, repr.plot.width=7)

# Extract pseudotime values from Monocle3  
cds$monocle3_pseudotime <- pseudotime(cds)

# Convert cell metadata into a dataframe  
data.pseudo <- as.data.frame(colData(cds))

# Generate a boxplot showing pseudotime distribution across cell types  
scPlot <- ggplot(data.pseudo, aes(monocle3_pseudotime, 
                                  reorder(seurat_annotations, monocle3_pseudotime),  # Order by pseudotime  
                                  fill = seurat_annotations)) +  # Color by cell type  
          geom_boxplot()  # Create boxplot  

# Display the boxplot  
scPlot  

# Optional: Save the plot as an image  
# ggsave("07-boxplot.png", plot = scPlot, bg = "white")

最后，可以检查若干基因的表达如何随伪时间变化。


In [ ]:
# Extract expression data for selected genes (CD44 and CXCL2)  
cds_subset <- cds[c('CD44', 'CXCL2'), ]

In [ ]:
# Generate a plot showing expression of selected genes across pseudotime  
scPlot <- plot_genes_in_pseudotime(cds_subset)

# Display the plot  
scPlot  

# Optional: Save the plot as an image  
# ggsave("08-genes_in_pseudotime.png", plot = scPlot, bg = "white")

In [ ]:
# The code below identifies genes that change their expression over pseudotime.  
# However, running this can be time-consuming.  

# cds_pr_test_res <- graph_test(cds, neighbor_graph="principal_graph", cores=4)  # Perform differential expression analysis  
# pr_deg_ids <- row.names(subset(cds_pr_test_res, q_value < 0.05))  # Select significant genes based on q-value  

## 思考题：

- 选择不同 root cells 会如何影响分析？
- 对某个感兴趣的细胞类型执行该分析。你能识别出细胞亚型吗？
- 这个细胞类型的 marker gene 是否会随伪时间变化？
